# Register Effects on Dependency-Length Tail Shape

**Evaluation Artifact Demo**

This notebook demonstrates the evaluation of register effects (spoken vs. written) on dependency-distance tail shape across UD treebanks.

## What This Artifact Does

- Loads dependency-distance data from Universal Dependencies treebanks
- Fits Generalized Pareto Distribution (GPD) and power-law models to the tail of distance distributions
- Compares tail shapes (shape parameter xi) between matched spoken/written register pairs
- Tests whether spoken language has lighter-tailed distributions than written language
- Stratifies results by language family, treebank size, sentence length, and dependency relation type

## Data

The demo uses a small curated subset (3 sentences from Slovenian treebanks). For the full analysis, replace with `full_data_out.json` (33,030 sentences across 18 treebanks).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from loguru import logger

# Configure logger
logger.remove()
logger.add(lambda msg: print(msg, end=''), level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# GitHub data URL (will be pushed after notebook creation)
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-544c17-language-minimizes-dependency-distance/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL or local file."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.info(f"GitHub load failed: {e}, trying local file...")
    
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local file")

# Load data
raw_data = load_data()
logger.info(f"Loaded data with {len(raw_data['datasets'][0]['examples'])} examples")

## Configuration

Demo parameters set to minimal values for fast execution. Adjust these to scale up:
- `N_BOOT`: Number of bootstrap resamples for confidence intervals
- `THRESHOLD_Q`: Quantile threshold for tail definition (0.5 = median, used for quick demo)
- `LIMIT_EXAMPLES`: Max examples to use (set to None for all)

In [ ]:
# =========================
# DEMO CONFIGURATION
# =========================
# Set these to MINIMUM values for quick demo. Increase gradually for full analysis.

N_BOOT = 50              # Bootstrap resamples for CI (demo: 50, full: 300)
THRESHOLD_Q = 0.50      # Tail threshold quantile (demo: 0.50, full: 0.90)
LIMIT_EXAMPLES = None    # Max examples to load (demo: None=all 3, full: None)

RNG = np.random.default_rng(42)

# Matched register pairs (from artifact spec)
MATCHED_PAIRS = {
    "slovenian": {"spoken": "sl_sst", "written": "sl_ssj", "family": "Indo-European (Slavic)"},
    "french": {"spoken": "fr_rhapsodie", "written": "fr_gsd", "family": "Indo-European (Romance)"},
    "english": {"spoken": "en_eslspok", "written": "en_ewt", "family": "Indo-European (Germanic)"},
    "turkish": {"spoken": "tr_atis", "written": "tr_imst", "family": "Turkic"},
}

MAJOR_DEPRELS = [
    "nsubj", "obj", "iobj", "obl", "nmod", "amod", "advmod",
    "acl", "advcl", "conj", "compound", "case", "mark", "det",
]

FLAT_LIST_DEPRELS = {"flat", "list"}

print(f"Config: N_BOOT={N_BOOT}, THRESHOLD_Q={THRESHOLD_Q}, LIMIT_EXAMPLES={LIMIT_EXAMPLES}")

## Data Preparation

Parse raw examples into two DataFrames:
- **sentence_df**: One row per sentence with aggregated metrics
- **arc_df**: One row per dependency arc with individual distance measurements

In [ ]:
def build_frames(examples):
    """Parse examples into sentence_df and arc_df."""
    sent_rows = []
    arc_rows = []
    
    for i, ex in enumerate(examples):
        try:
            inp = json.loads(ex["input"])
            out = json.loads(ex["output"])
        except (json.JSONDecodeError, KeyError) as e:
            logger.warning(f"Skipping malformed example {i}: {e}")
            continue
        
        tb = ex["metadata_treebank_id"]
        reg = ex["metadata_register"]
        lang = ex["metadata_language"]
        fam = ex["metadata_language_family"]
        slen = ex["metadata_sentence_length"]
        
        norm_dists = out.get("normalized_distances", [])
        raw_dists = out.get("dependency_distances", [])
        deprels = out.get("deprel", [])
        upos = inp.get("upos", [])
        
        sent_rows.append({
            "sent_id": i,
            "treebank_id": tb,
            "language": lang,
            "family": fam,
            "register": reg,
            "sentence_length": slen,
            "num_arcs": ex["metadata_num_arcs"],
            "mean_norm_dist": float(np.mean(norm_dists)) if norm_dists else np.nan,
            "has_intj": bool(set(upos) & {"INTJ"}),
            "norm_dists": norm_dists,
        })
        
        n = min(len(norm_dists), len(deprels))
        has_intj_arc = "INTJ" in upos
        for k in range(n):
            arc_rows.append({
                "sent_id": i,
                "treebank_id": tb,
                "language": lang,
                "family": fam,
                "register": reg,
                "sentence_length": slen,
                "norm_dist": norm_dists[k],
                "raw_dist": raw_dists[k] if k < len(raw_dists) else np.nan,
                "deprel": deprels[k],
                "sent_has_intj": has_intj_arc,
            })
    
    sent_df = pd.DataFrame(sent_rows)
    arc_df = pd.DataFrame(arc_rows)
    logger.info(f"Built sentence_df={len(sent_df)} rows, arc_df={len(arc_df)} rows")
    return sent_df, arc_df

# Build frames
examples = raw_data["datasets"][0]["examples"]
if LIMIT_EXAMPLES is not None:
    examples = examples[:LIMIT_EXAMPLES]

sent_df, arc_df = build_frames(examples)
print(f"\nDataframes ready:")
print(f"  Sentences: {len(sent_df)}")
print(f"  Arcs: {len(arc_df)}")
print(f"  Treebanks: {sent_df['treebank_id'].nunique()}")

## Tail Distribution Fitting

Fit Generalized Pareto Distribution (GPD) to the tail of dependency-distance distributions. This captures whether spoken language has lighter-tailed distributions (smaller xi values indicate lighter tails).

In [ ]:
def mean_residual_life(x, quantiles):
    """Compute mean residual life curve for threshold selection."""
    out = []
    for q in quantiles:
        u = float(np.quantile(x, q))
        excess = x[x > u] - u
        if len(excess) < 5:
            continue
        out.append({
            "quantile": float(q),
            "threshold": u,
            "mean_excess": float(np.mean(excess)),
            "n_exceed": int(len(excess))
        })
    return out

def fit_gpd(x, threshold_q, n_boot=50, rng=None):
    """Fit Generalized Pareto Distribution to tail."""
    rng = rng or np.random.default_rng(0)
    u = float(np.quantile(x, threshold_q))
    excess = x[x > u] - u
    excess = excess[excess > 0]
    
    if len(excess) < 15:
        return {"status": "insufficient_exceedances", "n_exceed": int(len(excess))}
    
    try:
        xi, _loc, sigma = stats.genpareto.fit(excess, floc=0)
    except (RuntimeError, ValueError) as e:
        return {"status": f"fit_failed: {e}"}
    
    loglik = float(np.sum(stats.genpareto.logpdf(excess, xi, loc=0, scale=sigma)))
    aic = 2 * 2 - 2 * loglik
    
    # Bootstrap for CI
    n = len(excess)
    boot_xi = np.empty(n_boot)
    for b in range(n_boot):
        sample = rng.choice(excess, size=n, replace=True)
        try:
            bxi, _, _ = stats.genpareto.fit(sample, floc=0)
            boot_xi[b] = bxi
        except (RuntimeError, ValueError):
            boot_xi[b] = np.nan
    
    boot_xi = boot_xi[~np.isnan(boot_xi)]
    ci_lo, ci_hi = (
        (float(np.percentile(boot_xi, 2.5)), float(np.percentile(boot_xi, 97.5)))
        if len(boot_xi) > 5 else (float('nan'), float('nan'))
    )
    
    return {
        "status": "ok",
        "threshold_quantile": float(threshold_q),
        "threshold": u,
        "n_exceed": int(n),
        "xi": float(xi),
        "sigma": float(sigma),
        "loglik": loglik,
        "aic": float(aic),
        "xi_ci95_lo": ci_lo,
        "xi_ci95_hi": ci_hi,
    }

# Fit GPD for each treebank
logger.info("Fitting GPD tail models per treebank...")
treebank_models = {}
for tb, grp in sent_df.groupby("treebank_id"):
    norm = np.concatenate([np.asarray(v, dtype=float) for v in grp["norm_dists"] if len(v) > 0])
    norm = norm[np.isfinite(norm) & (norm > 0)]
    
    if len(norm) > 0:
        n_boot_tb = N_BOOT if len(norm) > 200 else max(20, N_BOOT // 2)
        gpd = fit_gpd(norm, THRESHOLD_Q, n_boot=n_boot_tb, rng=RNG)
        gpd["n_normalized_distances"] = int(len(norm))
        treebank_models[tb] = gpd
    else:
        treebank_models[tb] = {"status": "no_data"}

print(f"\nTail model results:")
for tb, m in treebank_models.items():
    status = m.get("status", "ok")
    if status == "ok":
        print(f"  {tb:20s}: xi={m['xi']:7.4f}, n_exceed={m['n_exceed']:6d}")
    else:
        print(f"  {tb:20s}: {status}")

## Main Analysis

Compute headline statistics:
1. Language-level tally: xi and mean dependency distance (MDD) per register pair
2. Summary statistics: counts of pairs showing lighter-tailed spoken vs. written language

In [ ]:
# Language-level tally: xi and MDD per pair
logger.info("Computing language-level statistics...")
tally = {}
xi_vals, alpha_vals = [], []

for name, spec in MATCHED_PAIRS.items():
    sp, wr = treebank_models.get(spec["spoken"], {}), treebank_models.get(spec["written"], {})
    sp_xi = sp.get("xi") if sp.get("status") == "ok" else None
    wr_xi = wr.get("xi") if wr.get("status") == "ok" else None
    
    sp_mdd = float(sent_df.loc[sent_df.treebank_id == spec["spoken"], "mean_norm_dist"].mean())
    wr_mdd = float(sent_df.loc[sent_df.treebank_id == spec["written"], "mean_norm_dist"].mean())
    
    entry = {
        "family": spec["family"],
        "xi_spoken": sp_xi,
        "xi_written": wr_xi,
        "xi_direction": None,
        "mdd_spoken": sp_mdd,
        "mdd_written": wr_mdd,
        "mdd_direction": "spoken>written" if sp_mdd > wr_mdd else "spoken≤written",
    }
    
    if sp_xi is not None and wr_xi is not None:
        entry["xi_direction"] = "spoken<written (lighter)" if sp_xi < wr_xi else "spoken≥written (heavier)"
        xi_vals.append(sp_xi)
        xi_vals.append(wr_xi)
    
    tally[name] = entry

# Summary counts
n_lighter = sum(1 for e in tally.values() if e["xi_direction"] == "spoken<written (lighter)")
n_heavier = sum(1 for e in tally.values() if e["xi_direction"] == "spoken≥written (heavier)")
n_higher_mdd = sum(1 for e in tally.values() if e["mdd_direction"] == "spoken>written")

print(f"\nLanguage-level summary:")
print(f"  Pairs with spoken lighter tail (xi): {n_lighter}")
print(f"  Pairs with spoken heavier tail (xi): {n_heavier}")
print(f"  Pairs with spoken higher MDD: {n_higher_mdd}")

## Results Summary

Display headline statistics and create a visualization of tail shapes across treebanks.

In [ ]:
# Create summary table
summary_rows = []
for name, entry in tally.items():
    summary_rows.append({
        "Pair": name,
        "Family": entry["family"],
        "xi_spoken": f"{entry['xi_spoken']:.4f}" if entry['xi_spoken'] is not None else "N/A",
        "xi_written": f"{entry['xi_written']:.4f}" if entry['xi_written'] is not None else "N/A",
        "xi_direction": entry["xi_direction"] or "N/A",
        "MDD_spoken": f"{entry['mdd_spoken']:.4f}",
        "MDD_written": f"{entry['mdd_written']:.4f}",
        "MDD_direction": entry["mdd_direction"],
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*120)
print("LANGUAGE-LEVEL SUMMARY: Register Effects on Tail Shape (xi) and Mean Dependency Distance (MDD)")
print("="*120)
print(summary_df.to_string(index=False))
print("="*120)

# Headline statistics
print(f"\nHEADLINE STATISTICS:")
print(f"  Pairs with LIGHTER tail in spoken language (xi):     {n_lighter} / {len([e for e in tally.values() if e['xi_direction']])}")
print(f"  Pairs with HEAVIER tail in spoken language (xi):     {n_heavier} / {len([e for e in tally.values() if e['xi_direction']])}")
print(f"  Pairs with HIGHER mean distance in spoken (MDD):    {n_higher_mdd} / {len(tally)}")
print(f"\n  Note: Only 3 demo sentences. Full analysis uses 33,030 sentences across 18 treebanks.")

## Visualization

Plot tail shape parameter (xi) across treebanks. Negative xi indicates lighter (bounded) tails, positive xi indicates heavier (Pareto-like) tails.

In [ ]:
# Prepare data for visualization
tb_plot_data = []
for tb, m in treebank_models.items():
    if m.get("status") == "ok":
        # Infer register from treebank ID (simplified heuristic)
        reg = "spoken" if any(x in tb for x in ["sst", "rhapsodie", "eslspok", "atis"]) else "written"
        tb_plot_data.append({
            "treebank": tb,
            "xi": m["xi"],
            "register": reg,
            "ci_lo": m.get("xi_ci95_lo", np.nan),
            "ci_hi": m.get("xi_ci95_hi", np.nan),
        })

if tb_plot_data:
    plot_df = pd.DataFrame(tb_plot_data).sort_values("xi")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot points and error bars
    colors = {"spoken": "#e74c3c", "written": "#3498db"}
    for reg in ["spoken", "written"]:
        data = plot_df[plot_df["register"] == reg]
        if len(data) > 0:
            y_pos = np.arange(len(data))
            ax.scatter(data["xi"], y_pos, c=colors[reg], s=100, label=reg, alpha=0.7)
            
            # Error bars (if available)
            yerr_lo = data["ci_lo"].values
            yerr_hi = data["ci_hi"].values
            mask = ~(np.isnan(yerr_lo) | np.isnan(yerr_hi))
            if mask.any():
                ax.errorbar(
                    data.loc[mask, "xi"],
                    y_pos[mask],
                    xerr=[data.loc[mask, "xi"] - yerr_lo[mask], yerr_hi[mask] - data.loc[mask, "xi"]],
                    fmt="none", ecolor=colors[reg], alpha=0.3, capsize=3
                )
    
    ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5, label="xi=0 (exponential tail)")
    ax.set_yticks(np.arange(len(plot_df)))
    ax.set_yticklabels(plot_df["treebank"])
    ax.set_xlabel("GPD Shape Parameter (xi)", fontsize=12)
    ax.set_title("Tail Shape Across Treebanks\n(Negative xi = Bounded/Lighter tail, Positive xi = Heavy Pareto tail)", fontsize=12)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3, axis="x")
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nVisualization: {len(tb_plot_data)} treebanks plotted")
else:
    print("No treebank models with valid xi values for visualization.")

## Next Steps: Scaling to Full Analysis

To run the full evaluation:

1. **Scale parameters** in the Config cell:
   - `N_BOOT = 300` (bootstrap resamples for confidence intervals)
   - `THRESHOLD_Q = 0.90` (MRL-informed threshold for tail definition)
   - `LIMIT_EXAMPLES = None` (use all examples)

2. **Use full data**: Replace `mini_demo_data.json` with `full_data_out.json` in the data loading cell

3. **Extended analyses** (in full artifact):
   - Decile-level paired Wilcoxon/t-tests (corrects pseudo-replication)
   - Power-law model comparison via Akaike weights
   - Stratification by language family, treebank size, sentence length, deprel type
   - Annotation reliability cross-checks
   - Robustness checks (threshold sensitivity, excluding flat/list deprels)
   - Qualitative inspection of longest-distance arcs

**Expected runtime**:
- Demo (3 sentences): ~5 seconds
- Full (33,030 sentences, N_BOOT=300): ~10-15 minutes